In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from joblib import load
from scipy.stats import pearsonr
import numpy as np

# Load the dataframe from the joblib file
df_Addhealth = load('async_sync_results_AddHealth.joblib')
df_Banerjee = load('async_sync_results_Banerjee.joblib')

df = pd.concat([df_Addhealth, df_Banerjee], ignore_index=True)
df = df.dropna()
df.head()

In [ ]:
# Scatter plot of sym1 vs sym2, colored by T
plt.figure(figsize=(10, 8))

# Convert T to string to prevent legend truncation (0.05 → 0.5)
df['T_str'] = df['T'].apply(lambda x: f"{x:.2f}" if x < 1 else f"{x:.1f}")

sns.scatterplot(data=df, x='sym1', y='sym2', hue='T_str', size='number_of_nodes', palette='viridis')
#plt.title(r'$\Xi_{\mathrm{symmetric}}(RCS)$ vs $\Xi_{\mathrm{asymmetric}}(RCS)$, Colored by T, Sized by Number of Nodes', fontsize=16)
plt.xlabel(r'$\Xi_{\mathrm{symmetric}}(RCS)$', fontsize=16)
plt.ylabel(r'$\Xi_{\mathrm{asymmetric}}(RCS)$', fontsize=16)
plt.plot([df['sym1'].min(), df['sym1'].max()], [df['sym1'].min(), df['sym1'].max()], 'k--')

# Add Pearson r in a styled text box
corr, _ = pearsonr(df['sym1'], df['sym2'])
textbox = dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray', alpha=0.85)
plt.text(0.05, 0.95, f'Pearson $r = {corr:.2f}$', transform=plt.gca().transAxes,
         fontsize=14, verticalalignment='top', bbox=textbox)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.savefig('symmetric_asymmetric_scatter_plot.png', dpi=300, bbox_inches='tight')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from joblib import load
import numpy as np

# Load the node log DataFrame
df_nodes_Addhealth = load('async_sync_node_log_AddHealth.joblib')
df_nodes_Banerjee = load('async_sync_node_log_Banerjee.joblib')
df_nodes = pd.concat([df_nodes_Addhealth, df_nodes_Banerjee], ignore_index=True)

# Load the edge log DataFrame
df_edges_Addhealth = load('async_sync_edge_log_AddHealth.joblib')
df_edges_Banerjee = load('async_sync_edge_log_Banerjee.joblib')
df_edges = pd.concat([df_edges_Addhealth, df_edges_Banerjee], ignore_index=True)

# Get a random subsample of 10 percent for faster plotting
#df_nodes = df_nodes.sample(frac=0.01, random_state=42)
#df_edges = df_edges.sample(frac=0.01, random_state=42)


def annotate_facet_kde(data, **kwargs):
    """Add diagonal line and Pearson r to each facet."""
    ax = plt.gca()
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.6)
    x = data['CPC_sync'].astype(float)
    y = data['CPC_async'].astype(float)
    corr, _ = pearsonr(x, y)
    textbox = dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.85)
    ax.text(0.05, 0.95, f'Pearson $r = {corr:.2f}$',
            transform=ax.transAxes, fontsize=10, verticalalignment='top', bbox=textbox)


def kde_plot(data, **kwargs):
    """Draw a filled KDE contour plot for the facet."""
    x = data['CPC_sync'].astype(float)
    y = data['CPC_async'].astype(float)
    ax = plt.gca()
    sns.kdeplot(x=x, y=y, ax=ax, fill=True, color='darkblue', levels=8,
                thresh=0.1, bw_adjust=0.8, alpha=1.0)


# --- Node importance KDE plots ---
g_nodes = sns.FacetGrid(df_nodes, col='T', col_wrap=3, height=4, aspect=1)
g_nodes.map_dataframe(kde_plot)
g_nodes.map_dataframe(annotate_facet_kde)
g_nodes.set_titles("T = {col_name}")
g_nodes.set_axis_labels(r"$NI_{\mathrm{symmetric}}$", r"$NI_{\mathrm{asymmetric}}$", fontsize=12)
g_nodes.set(xlim=(0, 1), ylim=(0, 1))

for ax in g_nodes.axes.flat:
    ax.title.set_fontsize(14)
    ax.set_aspect('equal')

g_nodes.fig.subplots_adjust(top=0.93)
g_nodes.fig.suptitle("Node Importance: Symmetric vs Asymmetric Update by Threshold T", fontsize=16)
plt.savefig('sym_asym_node_importance_kde_plots.png', dpi=300, bbox_inches='tight')


# --- Edge importance KDE plots ---
g_edges = sns.FacetGrid(df_edges, col='T', col_wrap=3, height=4, aspect=1)
g_edges.map_dataframe(kde_plot)
g_edges.map_dataframe(annotate_facet_kde)
g_edges.set_titles("T = {col_name}")
g_edges.set_axis_labels(r"$TI_{\mathrm{symmetric}}$", r"$TI_{\mathrm{asymmetric}}$", fontsize=12)
g_edges.set(xlim=(0, 1), ylim=(0, 1))

for ax in g_edges.axes.flat:
    ax.title.set_fontsize(14)
    ax.set_aspect('equal')

g_edges.fig.subplots_adjust(top=0.93)
g_edges.fig.suptitle("Tie Importance: Symmetric vs Asymmetric Update by Threshold T", fontsize=16)
plt.savefig('sym_asym_edge_importance_kde_plots.png', dpi=300, bbox_inches='tight')